# Feature-map and qubit ablation, all 30 configurations (CPU is fine)

**Purpose.** The paper's screening rule was calibrated on one quantum kernel, the single-layer IQP map with a
linear chain of ZZ interactions on 8 qubits. This notebook recomputes the utility gap under alternative quantum
kernels for every configuration in the study, so the paper can say whether the rule still makes correct decisions
when the quantum kernel changes. Earlier ablations covered 24 configurations; this run covers all 30 with one
code path.

**Kernels evaluated.** All use the same seed-42 features as the main run.

| Name | Circuit | Qubits | Status |
|---|---|---|---|
| `IQP-1L` | Hadamards, `RZ(x_i)`, ZZ on adjacent pairs, one layer | 8 | the paper's kernel (reproduced as a check) |
| `IQP-2L` | the same layer applied twice | 8 | original ablation |
| `Angle` | `RY(x_i)` on each qubit, no entanglement | 8 | original ablation |
| `IQP-1L-4q` | `IQP-1L` on the first 4 of the 8 features | 4 | original ablation |
| `IQP-1L-6q` | `IQP-1L` on the first 6 of the 8 features | 6 | new |
| `IQP-1L-all` | as `IQP-1L` but ZZ on all 28 pairs instead of the 7 adjacent ones | 8 | new; tests the linear-chain choice |

Circuit definitions are copied from the original ablation notebooks (`week4_*.ipynb`). Using the first k features
for k qubits is exactly what those notebooks did; because PCA components and per-component scaling are nested, it
equals rerunning the pipeline with k components.

**What stays fixed.** The features, the 1000/200 split, the three classical baselines and their published best F1,
the classical alignment `A_C`, the SVM regularization `C = 1`, and the F1 metric (binary for two-class tasks, macro
otherwise). Only the quantum kernel changes, so every change in the gap is caused by the kernel.

**Contents.**
0. Setup
1. Reference values (published results and earlier ablations)
2. Feature maps and their self-check
3. Loading features, and the reproduction check against the published kernel
4. Running all kernels for all configurations
5. Cross-check against the earlier ablation results
6. Summary: does the screen hold under each kernel?
7. Notes
8. Matched alignment for the 4- and 6-qubit kernels (optional, adds a minute)

**Runtime.** About 10 to 15 minutes on a CPU runtime. Kernels are computed from simulated states (one circuit per
data point), which on an exact simulator gives the same values as one circuit per pair; Section 2 checks this for
every feature map. **Output.** `featuremap_ablation_all30.json` in `qml_project`. Upload it.

---
# Section 0. Setup

In [ ]:
!pip install pennylane pennylane-lightning scikit-learn numpy --quiet
import numpy as np, os, json, time, warnings
import pennylane as qml, sklearn
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from scipy.stats import pearsonr
warnings.filterwarnings('ignore')
print('scikit-learn', sklearn.__version__, '| pennylane', qml.__version__)

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
def find_file(f):
    for p in [os.path.join(SAVE_DIR, f), f]:
        if os.path.exists(p): return p
    return None
OUT = os.path.join(SAVE_DIR, 'featuremap_ablation_all30.json')

---
# Section 1. Reference values

`CANON` holds the published value for every configuration, taken from the paper's canonical results file
(`grid30_canonical.json`): block (main grid or held-out), metric, quantum F1, best classical F1, gap, and classical
alignment. `OLD` holds the F1 of the 24 configurations ablated earlier, from their `*_results_week4.json` files, for
the cross-check in Section 5.

In [ ]:
CANON = {
"ag_news": {
"block": "main",
"domain": "text",
"encoder": "BERT",
"metric": "macro",
"quantum_f1": 0.722223,
"best_classical_f1": 0.73596,
"gap": -0.013738,
"c_kta": 0.323067
},
"agnews_minilm": {
"block": "main",
"domain": "text",
"encoder": "MiniLM",
"metric": "macro",
"quantum_f1": 0.812118,
"best_classical_f1": 0.822764,
"gap": -0.010645,
"c_kta": 0.450769
},
"agnews_roberta": {
"block": "main",
"domain": "text",
"encoder": "RoBERTa",
"metric": "macro",
"quantum_f1": 0.806215,
"best_classical_f1": 0.806001,
"gap": 0.000214,
"c_kta": 0.380437
},
"cifar10": {
"block": "main",
"domain": "vision",
"encoder": "CLIP",
"metric": "binary",
"quantum_f1": 0.994975,
"best_classical_f1": 0.994975,
"gap": 0.0,
"c_kta": 0.708294
},
"fashion_clip": {
"block": "main",
"domain": "vision",
"encoder": "CLIP",
"metric": "binary",
"quantum_f1": 1.0,
"best_classical_f1": 1.0,
"gap": 0.0,
"c_kta": 0.826845
},
"fashion_dinov2": {
"block": "main",
"domain": "vision",
"encoder": "DINOv2",
"metric": "binary",
"quantum_f1": 1.0,
"best_classical_f1": 1.0,
"gap": 0.0,
"c_kta": 0.755055
},
"mscoco": {
"block": "main",
"domain": "multimodal",
"encoder": "CLIP",
"metric": "binary",
"quantum_f1": 0.753769,
"best_classical_f1": 0.807692,
"gap": -0.053923,
"c_kta": 0.017995
},
"mscoco_siglip": {
"block": "main",
"domain": "multimodal",
"encoder": "SigLIP",
"metric": "binary",
"quantum_f1": 0.798122,
"best_classical_f1": 0.809524,
"gap": -0.011402,
"c_kta": 0.018911
},
"multiling_mbert_arabic": {
"block": "held-out",
"domain": "multilingual",
"encoder": "mBERT",
"metric": "macro",
"quantum_f1": 0.391891,
"best_classical_f1": 0.468684,
"gap": -0.076793,
"c_kta": 0.024975
},
"multiling_mbert_chinese": {
"block": "held-out",
"domain": "multilingual",
"encoder": "mBERT",
"metric": "macro",
"quantum_f1": 0.501647,
"best_classical_f1": 0.573414,
"gap": -0.071767,
"c_kta": 0.052091
},
"multiling_mbert_english": {
"block": "held-out",
"domain": "multilingual",
"encoder": "mBERT",
"metric": "macro",
"quantum_f1": 0.513527,
"best_classical_f1": 0.51038,
"gap": 0.003147,
"c_kta": 0.032324
},
"multiling_mbert_german": {
"block": "held-out",
"domain": "multilingual",
"encoder": "mBERT",
"metric": "macro",
"quantum_f1": 0.509659,
"best_classical_f1": 0.522158,
"gap": -0.012499,
"c_kta": 0.106561
},
"multiling_mbert_hindi": {
"block": "held-out",
"domain": "multilingual",
"encoder": "mBERT",
"metric": "macro",
"quantum_f1": 0.322601,
"best_classical_f1": 0.39974,
"gap": -0.077138,
"c_kta": 0.004278
},
"multiling_mbert_spanish": {
"block": "held-out",
"domain": "multilingual",
"encoder": "mBERT",
"metric": "macro",
"quantum_f1": 0.391832,
"best_classical_f1": 0.427458,
"gap": -0.035626,
"c_kta": 0.045025
},
"multiling_xlmr_arabic": {
"block": "held-out",
"domain": "multilingual",
"encoder": "XLM-R",
"metric": "macro",
"quantum_f1": 0.581541,
"best_classical_f1": 0.571327,
"gap": 0.010214,
"c_kta": 0.077902
},
"multiling_xlmr_chinese": {
"block": "held-out",
"domain": "multilingual",
"encoder": "XLM-R",
"metric": "macro",
"quantum_f1": 0.657014,
"best_classical_f1": 0.706591,
"gap": -0.049577,
"c_kta": 0.117541
},
"multiling_xlmr_english": {
"block": "held-out",
"domain": "multilingual",
"encoder": "XLM-R",
"metric": "macro",
"quantum_f1": 0.457371,
"best_classical_f1": 0.525094,
"gap": -0.067723,
"c_kta": 0.055734
},
"multiling_xlmr_german": {
"block": "held-out",
"domain": "multilingual",
"encoder": "XLM-R",
"metric": "macro",
"quantum_f1": 0.558628,
"best_classical_f1": 0.57868,
"gap": -0.020051,
"c_kta": 0.095432
},
"multiling_xlmr_hindi": {
"block": "held-out",
"domain": "multilingual",
"encoder": "XLM-R",
"metric": "macro",
"quantum_f1": 0.372263,
"best_classical_f1": 0.416121,
"gap": -0.043858,
"c_kta": 0.003798
},
"multiling_xlmr_spanish": {
"block": "held-out",
"domain": "multilingual",
"encoder": "XLM-R",
"metric": "macro",
"quantum_f1": 0.449676,
"best_classical_f1": 0.437361,
"gap": 0.012315,
"c_kta": 0.059544
},
"snlive": {
"block": "main",
"domain": "multimodal",
"encoder": "CLIP",
"metric": "macro",
"quantum_f1": 0.269916,
"best_classical_f1": 0.311735,
"gap": -0.041819,
"c_kta": 0.002403
},
"snlive_siglip": {
"block": "main",
"domain": "multimodal",
"encoder": "SigLIP",
"metric": "macro",
"quantum_f1": 0.246229,
"best_classical_f1": 0.295001,
"gap": -0.048772,
"c_kta": 0.003499
},
"sst2_minilm": {
"block": "main",
"domain": "text",
"encoder": "MiniLM",
"metric": "binary",
"quantum_f1": 0.7343,
"best_classical_f1": 0.780488,
"gap": -0.046188,
"c_kta": 0.094392
},
"sst2_roberta": {
"block": "main",
"domain": "text",
"encoder": "RoBERTa",
"metric": "binary",
"quantum_f1": 0.644898,
"best_classical_f1": 0.685512,
"gap": -0.040614,
"c_kta": 0.012015
},
"sst2_bert": {
"block": "main",
"domain": "text",
"encoder": "BERT",
"metric": "binary",
"quantum_f1": 0.655022,
"best_classical_f1": 0.706422,
"gap": -0.0514,
"c_kta": 0.038991
},
"cifar10_dinov2": {
"block": "main",
"domain": "vision",
"encoder": "DINOv2",
"metric": "binary",
"quantum_f1": 1.0,
"best_classical_f1": 1.0,
"gap": 0.0,
"c_kta": 0.691496
},
"cifar10_vit": {
"block": "main",
"domain": "vision",
"encoder": "ViT",
"metric": "binary",
"quantum_f1": 0.995025,
"best_classical_f1": 0.995025,
"gap": 0.0,
"c_kta": 0.715422
},
"fashion_vit": {
"block": "main",
"domain": "vision",
"encoder": "ViT",
"metric": "binary",
"quantum_f1": 0.984772,
"best_classical_f1": 0.989899,
"gap": -0.005127,
"c_kta": 0.652706
},
"snlive_flava": {
"block": "main",
"domain": "multimodal",
"encoder": "FLAVA",
"metric": "macro",
"quantum_f1": 0.316955,
"best_classical_f1": 0.360988,
"gap": -0.044033,
"c_kta": 0.006765
},
"mscoco_flava": {
"block": "main",
"domain": "multimodal",
"encoder": "FLAVA",
"metric": "binary",
"quantum_f1": 0.742268,
"best_classical_f1": 0.801843,
"gap": -0.059575,
"c_kta": 0.018308
}
}

OLD = {
"ag_news": {
"Angle": 0.711585,
"IQP 2L": 0.723756,
"4q": 0.598007
},
"agnews_roberta": {
"Angle": 0.805275,
"IQP 2L": 0.764814,
"4q": 0.63303
},
"cifar10": {
"Angle": 0.989899,
"IQP 2L": 0.984772,
"4q": 0.989899
},
"fashion_clip": {
"Angle": 1.0,
"IQP 2L": 0.994975,
"4q": 1.0
},
"fashion_dinov2": {
"Angle": 1.0,
"IQP 2L": 1.0,
"4q": 1.0
},
"mscoco": {
"Angle": 0.766169,
"IQP 2L": 0.753927,
"4q": 0.724771
},
"mscoco_siglip": {
"Angle": 0.76699,
"IQP 2L": 0.759615,
"4q": 0.65625
},
"sst2_bert": {
"Angle": 0.696833,
"IQP 2L": 0.683544,
"4q": 0.659574
},
"snlive": {
"Angle": 0.291552,
"IQP 2L": 0.289653,
"4q": 0.302728
},
"snlive_siglip": {
"Angle": 0.281503,
"IQP 2L": 0.250567,
"4q": 0.342053
},
"sst2_minilm": {
"Angle": 0.792079,
"IQP 2L": 0.702439,
"4q": 0.757009
},
"sst2_roberta": {
"Angle": 0.636735,
"IQP 2L": 0.595745,
"4q": 0.587302
},
"multiling_mbert_english": {
"Angle": 0.510515,
"IQP 2L": 0.463851,
"4q": 0.365872
},
"multiling_mbert_spanish": {
"Angle": 0.41019,
"IQP 2L": 0.369371,
"4q": 0.390866
},
"multiling_mbert_german": {
"Angle": 0.504204,
"IQP 2L": 0.503476,
"4q": 0.48285
},
"multiling_mbert_arabic": {
"Angle": 0.461578,
"IQP 2L": 0.380415,
"4q": 0.337294
},
"multiling_mbert_chinese": {
"Angle": 0.560083,
"IQP 2L": 0.472575,
"4q": 0.412834
},
"multiling_mbert_hindi": {
"Angle": 0.282027,
"IQP 2L": 0.350995,
"4q": 0.340176
},
"multiling_xlmr_english": {
"Angle": 0.472294,
"IQP 2L": 0.445151,
"4q": 0.356889
},
"multiling_xlmr_spanish": {
"Angle": 0.434602,
"IQP 2L": 0.404803,
"4q": 0.397221
},
"multiling_xlmr_german": {
"Angle": 0.574344,
"IQP 2L": 0.563341,
"4q": 0.463052
},
"multiling_xlmr_arabic": {
"Angle": 0.532795,
"IQP 2L": 0.499632,
"4q": 0.552268
},
"multiling_xlmr_chinese": {
"Angle": 0.701668,
"IQP 2L": 0.637548,
"4q": 0.52519
},
"multiling_xlmr_hindi": {
"Angle": 0.331113,
"IQP 2L": 0.322076,
"4q": 0.305561
}
}

TAGS = list(CANON)
print(len(TAGS), 'configurations;', sum(v['block'] == 'main' for v in CANON.values()), 'main grid,',
      sum(v['block'] == 'held-out' for v in CANON.values()), 'held-out;', len(OLD), 'with earlier ablation results')

---
# Section 2. Feature maps

The four circuit families. `make_fast_kernel` builds a kernel function that simulates each input's state once and
returns |<phi(b)|phi(a)>|^2 for all pairs. The self-check compares it with the original per-pair circuit
(`U(a)` followed by `U(b)` adjoint, probability of all zeros) for every feature map and qubit count used here.

In [ ]:
def angle_map(x, wires):
    for i, w in enumerate(wires): qml.RY(x[i], wires=w)

def iqp_1layer(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]]); qml.RZ(x[i] * x[i+1], wires=wires[i+1]); qml.CNOT(wires=[wires[i], wires[i+1]])

def iqp_2layer(x, wires):
    for _ in range(2): iqp_1layer(x, wires)

def iqp_1layer_all(x, wires):   # new: ZZ on every pair instead of adjacent pairs only
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n):
        for j in range(i + 1, n):
            qml.CNOT(wires=[wires[i], wires[j]]); qml.RZ(x[i] * x[j], wires=wires[j]); qml.CNOT(wires=[wires[i], wires[j]])

DEVICES = {n: qml.device('lightning.qubit', wires=n) for n in (4, 6, 8)}

def make_fast_kernel(fmap, n):
    @qml.qnode(DEVICES[n])
    def state(x):
        fmap(x, wires=range(n)); return qml.state()
    def K(A, B=None):
        SA = np.array([state(a[:n]) for a in A]); SB = SA if B is None else np.array([state(b[:n]) for b in B])
        return np.abs(SA.conj() @ SB.T) ** 2
    return K

def make_pair_kernel(fmap, n):
    @qml.qnode(DEVICES[n])
    def circ(a, b):
        fmap(a, wires=range(n)); qml.adjoint(fmap)(b, wires=range(n)); return qml.probs(wires=range(n))
    return lambda a, b: float(circ(a[:n], b[:n])[0])

MAPS = {   # name: (feature map, number of qubits)
    'IQP-1L':     (iqp_1layer, 8),
    'IQP-2L':     (iqp_2layer, 8),
    'Angle':      (angle_map, 8),
    'IQP-1L-4q':  (iqp_1layer, 4),
    'IQP-1L-6q':  (iqp_1layer, 6),
    'IQP-1L-all': (iqp_1layer_all, 8)}
FAST = {k: make_fast_kernel(*v) for k, v in MAPS.items()}
PAIR = {k: make_pair_kernel(*v) for k, v in MAPS.items()}

_A = np.random.RandomState(3).uniform(0, np.pi, (12, 8))
for name in MAPS:
    slow = np.array([[PAIR[name](_A[i], _A[j]) for j in range(12)] for i in range(12)])
    fast = FAST[name](_A)
    assert np.allclose(np.diag(slow), 1.0), name
    assert np.abs(fast - slow).max() < 1e-10, name
    print(f'  {name:<11} fast kernel equals per-pair circuit (max diff {np.abs(fast - slow).max():.1e})')
print('Section 2 self-check passed')

---
# Section 3. Features and reproduction check

For each configuration this loads the seed-42 features saved by the main run, recomputes the paper's `IQP-1L`
kernel, and compares it entry by entry with the kernel saved on Drive. It then refits the quantum SVM and the three
classical SVMs and compares their F1 with the published values. A configuration is `PASS` when the kernel agrees to
within 1e-4 and both F1 values agree exactly. Only `PASS` configurations are used in Section 4.

SST-2 with BERT was the first configuration run; its saved kernel file holds the training features, and its test
features were recovered and verified by `sst2_bert_complete.ipynb`, so they are read from `sst2_bert_test_labels.npz`.

In [ ]:
def load_features(tag):
    if tag == 'sst2_bert':
        z = np.load(find_file('kernel_matrices_week3.npz')); t = np.load(find_file('sst2_bert_test_labels.npz'))
        return z['X_tr'].astype(np.float64), z['y_tr'], t['X_te'].astype(np.float64), t['y_te']
    for f, keys in [(f'{tag}_embeddings.npz', ('X_tr_subset', 'y_tr_subset', 'X_te_subset', 'y_te_subset')),
                    (f'{tag}_kernel_matrices.npz', ('X_tr', 'y_tr', 'X_te', 'y_te'))]:
        p = find_file(f)
        if p:
            z = np.load(p)
            if all(k in z.files for k in keys):
                return z[keys[0]].astype(np.float64)[:1000], z[keys[1]][:1000], z[keys[2]].astype(np.float64), z[keys[3]]
    return None

def saved_kernel(tag):
    p = find_file('kernel_matrices_week3.npz' if tag == 'sst2_bert' else f'{tag}_kernel_matrices.npz')
    if p is None: return None
    z = np.load(p); return z['K_train_quantum'], z['K_test_quantum']

def f1_for(tag):
    avg = 'binary' if CANON[tag]['metric'] == 'binary' else 'macro'
    return lambda y, p: float(f1_score(y, p, average=avg))

def classical_best(tag, X_tr, y_tr, X_te, y_te):
    f1 = f1_for(tag); out = {}
    for name, clf in [('Linear', SVC(kernel='linear', C=1.0, random_state=42)),
                      ('RBF', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)),
                      ('Polynomial', SVC(kernel='poly', degree=3, coef0=1, C=1.0, gamma='scale', random_state=42))]:
        out[name] = f1(y_te, clf.fit(X_tr, y_tr).predict(X_te))
    return max(out.values())

DATA, STATUS = {}, {}
for tag in TAGS:
    feats = load_features(tag); ks = saved_kernel(tag)
    if feats is None or ks is None:
        STATUS[tag] = 'NO INPUT'; print(f'{tag:<26} NO INPUT (features or saved kernel missing on Drive)'); continue
    X_tr, y_tr, X_te, y_te = feats
    K_tr, K_te = FAST['IQP-1L'](X_tr), FAST['IQP-1L'](X_te, X_tr)
    kd = max(np.abs(K_tr - ks[0]).max(), np.abs(K_te - ks[1]).max())
    f1 = f1_for(tag)
    qf1 = f1(y_te, SVC(kernel='precomputed', C=1.0).fit(K_tr, y_tr).predict(K_te))
    cf1 = classical_best(tag, X_tr, y_tr, X_te, y_te)
    dq, dc = qf1 - CANON[tag]['quantum_f1'], cf1 - CANON[tag]['best_classical_f1']
    ok = kd < 1e-4 and abs(dq) < 1e-6 and abs(dc) < 1e-6
    STATUS[tag] = 'PASS' if ok else 'FAIL'
    if ok: DATA[tag] = (X_tr, y_tr, X_te, y_te)
    print(f'{tag:<26} {STATUS[tag]}  kernel max diff {kd:.1e} | quantum F1 diff {dq:+.1e} | best classical F1 diff {dc:+.1e}')
print(f'\n{sum(s == "PASS" for s in STATUS.values())} of {len(TAGS)} configurations reproduce the published results')

---
# Section 4. All kernels for all configurations

For every configuration that passed Section 3, each of the six kernels is computed on the identical features. For
each kernel the cell records the quantum F1, the utility gap against the published best classical F1, the quantum
kernel-target alignment `A_Q`, and the mean off-diagonal kernel entry. Results are written to Drive after every
configuration, and configurations already in the output file are skipped, so the cell can be rerun safely.

In [ ]:
def kta(K, y):
    y = np.asarray(y); n = len(y); Y = np.zeros((n, n))
    for c in np.unique(y): m = (y == c).astype(float); Y += np.outer(m, m)
    H = np.eye(n) - np.ones((n, n)) / n; Kc, Yc = H @ K @ H, H @ Y @ H
    return float(np.sum(Kc * Yc) / np.sqrt(np.sum(Kc * Kc) * np.sum(Yc * Yc)))

RESULTS = json.load(open(OUT)) if os.path.exists(OUT) else {}
t_all = time.time()
for tag, (X_tr, y_tr, X_te, y_te) in DATA.items():
    if tag in RESULTS and all(m in RESULTS[tag]['kernels'] for m in MAPS):
        print(f'{tag:<26} already done'); continue
    t0 = time.time(); f1 = f1_for(tag); best_c = CANON[tag]['best_classical_f1']; off = ~np.eye(len(X_tr), dtype=bool); res = {}
    for name in MAPS:
        K_tr, K_te = FAST[name](X_tr), FAST[name](X_te, X_tr)
        q = f1(y_te, SVC(kernel='precomputed', C=1.0).fit(K_tr, y_tr).predict(K_te))
        res[name] = {'quantum_f1': q, 'gap': q - best_c, 'kta': kta(K_tr, y_tr), 'offdiag': float(K_tr[off].mean())}
    RESULTS[tag] = {**CANON[tag], 'kernels': res}
    json.dump(RESULTS, open(OUT, 'w'), indent=1)
    print(f'{tag:<26} ' + '  '.join(f'{n} {res[n]["gap"]:+.3f}' for n in MAPS) + f'   ({time.time()-t0:.0f} s)')
print(f'\n{len(RESULTS)} configurations saved to {OUT} ({(time.time()-t_all)/60:.1f} min)')

---
# Section 5. Cross-check against the earlier ablation results

The earlier ablations computed `Angle`, `IQP-2L`, and the 4-qubit kernel one circuit per pair for 24
configurations. Recomputing them here must give the same F1, because the circuits and features are identical. Any
difference larger than float noise means something has changed and must be explained before the results are used.

In [ ]:
print(f"{'configuration':<26} {'Angle':>9} {'IQP-2L':>9} {'4 qubits':>9}   (new F1 minus earlier F1)")
worst = 0.0
for tag, old in OLD.items():
    if tag not in RESULTS: print(f'{tag:<26} not run'); continue
    k = RESULTS[tag]['kernels']
    d = [k['Angle']['quantum_f1'] - old['Angle'], k['IQP-2L']['quantum_f1'] - old['IQP 2L'], k['IQP-1L-4q']['quantum_f1'] - old['4q']]
    worst = max(worst, max(abs(x) for x in d))
    print(f'{tag:<26} ' + ' '.join(f'{x:>+9.1e}' for x in d))
print(f'\nlargest difference {worst:.1e}:', 'AGREES with the earlier ablations' if worst < 1e-6 else 'DIFFERS, investigate before use')

---
# Section 6. Does the screen hold under each kernel?

For each kernel this applies the paper's rule, skip the quantum kernel when the classical alignment `A_C` is below
0.05. `A_C` does not depend on the quantum kernel, so the same configurations are flagged under every kernel. The
table reports, separately for the 18-configuration main grid and the 12-configuration hold-out, the correlation
between `A_C` and the gap, the number flagged, the largest gap among flagged configurations (the most a practitioner
following the rule would have given up), and how many configurations anywhere beat the best classical kernel by
more than 0.005.

In [ ]:
def screen(block):
    tags = [t for t in RESULTS if RESULTS[t]['block'] == block]
    a = np.array([RESULTS[t]['c_kta'] for t in tags]); flag = a < 0.05
    print(f'\n{block} ({len(tags)} configurations, {flag.sum()} flagged at A_C < 0.05)')
    print(f"  {'kernel':<11} {'r(A_C, gap)':>12} {'p':>8} {'flagged mean':>13} {'flagged max':>12} {'wins > 0.005':>13}")
    for name in MAPS:
        g = np.array([RESULTS[t]['kernels'][name]['gap'] for t in tags]); r, p = pearsonr(a, g)
        print(f'  {name:<11} {r:>12.3f} {p:>8.3f} {g[flag].mean():>+13.3f} {g[flag].max():>+12.3f} {int((g > 0.005).sum()):>13}')
screen('main'); screen('held-out')

print(f"\nPer-configuration gaps\n{'configuration':<26} {'A_C':>6} " + ' '.join(f'{n:>11}' for n in MAPS))
for t in sorted(RESULTS, key=lambda t: (RESULTS[t]['block'] != 'main', RESULTS[t]['domain'], t)):
    print(f'{t:<26} {RESULTS[t]["c_kta"]:>6.3f} ' + ' '.join(f'{RESULTS[t]["kernels"][n]["gap"]:>+11.3f}' for n in MAPS))

---
# Section 7. Notes

**FAIL in Section 3.** The features on Drive no longer reproduce the published kernel or F1. Report the printed
differences; do not use that configuration until it is explained.

**NO INPUT in Section 3.** The configuration's `<tag>_embeddings.npz` or `<tag>_kernel_matrices.npz` is not in
`qml_project`. Copy it from the original run.

**DIFFERS in Section 5.** The recomputed ablation does not match the earlier run. The circuits are copied from those
notebooks, so a difference would point to different features or a different metric. Report it.

**The C sweep.** The earlier ablation notebooks also ran a regularization sweep, but on a 200-point subset rather than
the full 1000 points, so it is not comparable with the main results and is not repeated or reported here.

**Hand-off.** Upload `featuremap_ablation_all30.json`. It replaces the earlier ablation numbers in the paper (Table 5
and the per-configuration appendix table) with results for all 30 configurations from one verified code path.

---
# Section 8. Matched alignment for the 4- and 6-qubit kernels (optional)

The 4- and 6-qubit kernels see only the first 4 or 6 of the eight scaled features, but the screen's alignment `A_C`
is computed on all eight. If the screen is applied the way a practitioner would apply it to a smaller kernel, with
alignment computed on the features that kernel actually receives (an RBF kernel with gamma = 1/k on the first k
features), does it recover? For k = 8 this must equal the published `A_C`, which the cell asserts before reporting
anything. Needs Sections 0 to 4 to have run in this session.

In [ ]:
def matched_alignment(X_tr, y_tr, k):
    from sklearn.metrics.pairwise import rbf_kernel
    return kta(rbf_kernel(X_tr[:, :k], X_tr[:, :k], gamma=1.0 / k), y_tr)

# unit test: with k = 8 the matched alignment is the published A_C
for tag, (X_tr, y_tr, X_te, y_te) in DATA.items():
    assert abs(matched_alignment(X_tr, y_tr, 8) - CANON[tag]['c_kta']) < 1e-4, tag
print('check passed: k = 8 reproduces the published A_C for', len(DATA), 'configurations')

MATCHED = {tag: {k: matched_alignment(DATA[tag][0], DATA[tag][1], k) for k in (4, 6)} for tag in DATA}
for block in ('main', 'held-out'):
    tags = [t for t in RESULTS if RESULTS[t]['block'] == block and t in MATCHED]
    if len(tags) < 3: print(f'{block}: too few configurations to correlate ({len(tags)})'); continue
    print(f'\n{block} ({len(tags)} configurations)')
    print(f"  {'kernel':<11} {'alignment':<10} {'r':>7} {'p':>8} {'flagged':>8} {'worst flagged':>14}")
    for name, k in (('IQP-1L-6q', 6), ('IQP-1L-4q', 4)):
        g = np.array([RESULTS[t]['kernels'][name]['gap'] for t in tags])
        for label, a in (('all 8', np.array([RESULTS[t]['c_kta'] for t in tags])), (f'first {k}', np.array([MATCHED[t][k] for t in tags]))):
            r, p = pearsonr(a, g); fl = a < 0.05
            print(f'  {name:<11} {label:<10} {r:>7.3f} {p:>8.4f} {int(fl.sum()):>8} {g[fl].max() if fl.any() else float("nan"):>+14.3f}')